# Qwen2.5-VL-7B-Instruct-AWQ Demo

This notebook demonstrates how to use the Qwen2.5-VL-7B-Instruct-AWQ vision-language model served via vLLM with an OpenAI-compatible API.

## Setup

Make sure the vLLM server is running:
```bash
./start_qwen_vl_server.sh
```

In [ ]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="unused")

# Auto-detect the served model name
MODEL = client.models.list().data[0].id
print(f"Connected to model: {MODEL}")

## Example 1: Simple Chat Completion

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is the capital of France? Answer briefly."},
    ],
    max_tokens=128,
    temperature=0.7,
)

print("Response:", response.choices[0].message.content)
print(f"Tokens: {response.usage.prompt_tokens} prompt + {response.usage.completion_tokens} completion")

## Example 2: Streaming Response

In [ ]:
stream = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Write a haiku about programming."},
    ],
    max_tokens=64,
    temperature=0.7,
    stream=True,
)

print("Streaming response:")
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
print()

## Example 3: Multi-turn Conversation

In [ ]:
messages = [
    {"role": "system", "content": "You are a robotics expert."},
    {"role": "user", "content": "What are the main sensors used in autonomous robots?"},
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    max_tokens=256,
    temperature=0.7,
)

assistant_reply = response.choices[0].message.content
print("Assistant:", assistant_reply)

In [ ]:
# Continue the conversation
messages.append({"role": "assistant", "content": assistant_reply})
messages.append({"role": "user", "content": "Which of those is most important for indoor navigation?"})

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    max_tokens=256,
    temperature=0.7,
)

print("Assistant:", response.choices[0].message.content)

## Example 4: Structured JSON Output

In [ ]:
import json

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant that responds only in valid JSON."},
        {"role": "user", "content": "List 3 common household objects with their typical dimensions in cm. Use format: [{\"object\": ..., \"width\": ..., \"height\": ..., \"depth\": ...}]"},
    ],
    max_tokens=256,
    temperature=0.3,
)

raw = response.choices[0].message.content
print("Raw response:")
print(raw)

try:
    parsed = json.loads(raw)
    print("\nParsed JSON:")
    print(json.dumps(parsed, indent=2))
except json.JSONDecodeError as e:
    print(f"\nFailed to parse JSON: {e}")

## Example 5: Adjusting Generation Parameters

In [ ]:
prompt = "Explain what a LiDAR sensor does in one paragraph."

for temp in [0.1, 0.7, 1.2]:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=128,
        temperature=temp,
    )
    print(f"--- Temperature {temp} ---")
    print(response.choices[0].message.content)
    print()

## Example 6: Image Description (Local File)

In [ ]:
import base64
from PIL import Image
from IPython.display import display

# Load and display a local image
local_image_path = "room_expo.png"
image = Image.open(local_image_path)
display(image)

# Encode image to base64
with open(local_image_path, "rb") as f:
    img_b64 = base64.b64encode(f.read()).decode()

# Send image to the model
response = client.chat.completions.create(
    model=MODEL,
    messages=[{
        "role": "user",
        "content": [
            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_b64}"}},
            {"type": "text", "text": "What objects do you see in this image? List them briefly."},
        ],
    }],
    max_tokens=256,
)

print("Model Response:")
print(response.choices[0].message.content)

## Example 7: Structured Object Detection from Image

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{
        "role": "user",
        "content": [
            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_b64}"}},
            {"type": "text", "text": (
                "Create a JSON list for each object in the image in the format "
                '{"bbox_2d": [x1, y1, x2, y2], "label": "object_name"}. '
                "If there are multiple objects with the same label, add numerals: "
                '"object1", "object2", etc. Return only valid JSON.'
            )},
        ],
    }],
    max_tokens=512,
    temperature=0.3,
)

raw = response.choices[0].message.content
print("Raw response:")
print(raw)

try:
    objects = json.loads(raw)
    print(f"\nDetected {len(objects)} objects:")
    for obj in objects:
        print(f"  {obj['label']}: {obj['bbox_2d']}")
except json.JSONDecodeError as e:
    print(f"\nFailed to parse JSON: {e}")

## Example 8: Image from URL

In [ ]:
image_url = "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"

response = client.chat.completions.create(
    model=MODEL,
    messages=[{
        "role": "user",
        "content": [
            {"type": "image_url", "image_url": {"url": image_url}},
            {"type": "text", "text": "Describe this image in detail."},
        ],
    }],
    max_tokens=256,
)

print("Model Response:")
print(response.choices[0].message.content)